In [2]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from sklearn import metrics, clone
from sklearn.model_selection import GridSearchCV, train_test_split,StratifiedKFold,cross_val_predict
from sklearn.metrics import matthews_corrcoef, make_scorer,auc, accuracy_score, recall_score,roc_curve, roc_auc_score,precision_score,f1_score
from sklearn.model_selection import learning_curve

import joblib

import matplotlib.pyplot as plt

from warnings import filterwarnings
# Silence some expected warnings
filterwarnings("ignore")

In [3]:
folder_path = r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results"

In [4]:
def load_dataset(seed):
    if seed == 8:
        tr_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\tr_8_ecfp.csv')
        te_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\te_8_ecfp.csv')

    if seed == 16:
        tr_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\tr_16_ecfp.csv')
        te_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\te_16_ecfp.csv')

    if seed == 24:
        tr_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\tr_24_ecfp.csv')
        te_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\te_24_ecfp.csv')

    if seed == 32:
        tr_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\tr_32_ecfp.csv')
        te_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\te_32_ecfp.csv')

    if seed == 48:
        tr_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\tr_48_ecfp.csv')
        te_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\te_48_ecfp.csv')

    if seed == 64:
        tr_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\tr_64_ecfp.csv')
        te_df = pd.read_csv(folder_path + f'\\{seed}\\' + '\\te_64_ecfp.csv')

    tr_x = tr_df.iloc[:,5:].values.astype(np.int32)
    tr_y = tr_df.iloc[:,4].values.astype(np.int32)

    te_x = te_df.iloc[:,5:].values.astype(np.int32)
    te_y = te_df.iloc[:,4].values.astype(np.int32)
    
    return tr_x,te_x,tr_y,te_y

In [5]:
def Grid_search(model_name,random_seed,tr_x,tr_y,fp):
    
    if model_name == 'RF':
        estimator = RandomForestClassifier(random_state=random_seed)
        param_grid = {
            'n_estimators': list(range(10, 101, 20)),          
            'max_leaf_nodes': [20, 30, 50],        
            'max_features': [1,10,20],            
        }

    elif model_name == 'XGBoost':
        estimator = xgb.XGBClassifier(random_state=random_seed)
        param_grid = {
            'learning_rate': [0.1, 0.01],    
            'n_estimators': list(range(10, 101, 20)), 
            'max_depth': [2, 3, 4],        
            'subsample': [0.3, 0.5],  
            'colsample_bytree': [0.3, 0.5]  
        }
        
    #mcc_scorer = make_scorer(matthews_corrcoef)
        
    #cv = StratifiedKFold(n_splits=fold, shuffle=True,random_state=random_seed) 
        
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring='accuracy', 
        cv=5,                      
        n_jobs=1,                        
        verbose=1)  
            

    grid_search.fit(tr_x, tr_y)
        
    best_params = grid_search.best_params_     # best hyperparameters
    best_score = grid_search.best_score_
    print(f'best_params: {best_params}')
    print(f'best_score: {best_score}')
    cv_resluts = pd.DataFrame(grid_search.cv_results_)    
        
    best_model = estimator.set_params(**best_params)
    best_model.fit(tr_x, tr_y)
    # best_model = grid_search.best_estimator_
        
    cv_resluts.to_csv(r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results\grid_search_{fp}_{model}_seed_{random_seed}.csv".format(fp=fp,model=model_name,random_seed=random_seed),index=False)
    
    joblib.dump(best_model,r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results\{fp}_{model}_seed_{random_seed}.joblib".format(fp=fp,model=model_name,random_seed=random_seed))
    
    return best_params,best_score,cv_resluts,best_model

In [6]:
def Predict_Evalute(best_model,tr_x,te_x,tr_y,te_y,random_seed,fp,model_name):
    
    #best_model.fit(tr_x, tr_y)
    
    tr_pred = best_model.predict(tr_x)
    tr_proba = best_model.predict_proba(tr_x)[:, 1]
    
    te_pred = best_model.predict(te_x)
    te_proba = best_model.predict_proba(te_x)[:, 1]
    
    tr_result = Performance(tr_pred,tr_proba,tr_y)
    te_result = Performance(te_pred,te_proba,te_y)
    
    pred_evaluation = pd.DataFrame([random_seed,
                                    tr_result[0],tr_result[1],tr_result[2],tr_result[3],tr_result[4],tr_result[5],tr_result[6],tr_result[7],tr_result[8],tr_result[9],tr_result[10],tr_result[11],
                                    te_result[0],te_result[1],te_result[2],te_result[3],te_result[4],te_result[5],te_result[6],te_result[7],te_result[8],te_result[9],te_result[10],te_result[11]],
                                    index=['split_seed',
                                           'tr_mcc','tr_accuracy','tr_auc','tr_f1','tr_pre','tr_se','tr_sp','tr_tp','tr_fn','tr_fp','tr_tn','tr_youden',
                                           'te_mcc','te_accuracy','te_auc','te_f1','te_pre','te_se','te_sp','te_tp','te_fn','te_fp','te_tn','te_youden']).T
    path = r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results\performance_result_{fp}_{model}_seed_{random_seed}.csv".format(fp=fp,model=model_name,random_seed=random_seed)
    pred_evaluation.to_csv(path,index=False)

    pred_results = pd.DataFrame([tr_y,tr_pred,tr_proba,te_y,te_pred,te_proba],
                                index=['tr_y','tr_pred','tr_proba','te_y','te_pred','te_proba']).T
            
    path = r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results\predict_values_{fp}_{model}_seed_{random_seed}.csv".format(fp=fp,model=model_name,random_seed=random_seed)
    pred_results.to_csv(path,index=False)
    
    #print(tr_result,cv_result,te_result)
    #ROC(tr_y,tr_proba,te_y,te_proba,fp,model_name,random_seed)
    
    return tr_result,te_result

In [7]:
def find_best_threshold(fpr, tpr, thresholds):
    J = tpr - fpr  # Youden's J statistic
    best_idx = np.argmax(J)  # Index of the maximum J value
    best_threshold = thresholds[best_idx]
    return best_threshold, fpr[best_idx], tpr[best_idx], J[best_idx]
    
def Performance(y_pred,y_proba,y_true):
    mcc = matthews_corrcoef(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    best_threshold, best_fpr, best_tpr, J = find_best_threshold(fpr, tpr, thresholds)
    
    tp = 0
    fn = 0
    fp = 0
    tn = 0
    for i in range(len(y_true)):
        if y_true[i] == 1 and y_pred[i] == 1:
            tp += 1
        if y_true[i] == 1 and y_pred[i] == 0:
            fn += 1
        if y_true[i] == 0 and y_pred[i] == 1:
            fp += 1
        if y_true[i] == 0 and y_pred[i] == 0:
            tn += 1

    # 计算Sensitivity
    se = round(float(tp) / float(tp + fn),4)
    # 计算Specificity
    sp = round(float(tn) / float(tn + fp),4)
    return mcc,accuracy,auc,f1,precision,se,sp,tp,fn,fp,tn,J

In [9]:
model_list = ['RF'] # ['RF','XGBoost']
for i in model_list:
    print(i)
    for random_seed in [8,16,24,32,48,64]:
        tr_x,te_x,tr_y,te_y = load_dataset(random_seed)
        best_params,best_score,cv_resluts,best_model = Grid_search(i,random_seed,tr_x,tr_y,fp='ecfp') # 'svm','rf','dt','xgb'
        tr_result,te_result = Predict_Evalute(best_model,tr_x,te_x,tr_y,te_y,random_seed,fp='ecfp',model_name = i)
        print(f'Finish random_seed {random_seed}')
    print(f'Finish model {i}')
print('Done')

RF
Fitting 5 folds for each of 45 candidates, totalling 225 fits
best_params: {'max_features': 20, 'max_leaf_nodes': 50, 'n_estimators': 90}
best_score: 0.7224169414638647
Finish random_seed 8
Fitting 5 folds for each of 45 candidates, totalling 225 fits
best_params: {'max_features': 20, 'max_leaf_nodes': 50, 'n_estimators': 90}
best_score: 0.7231883406370612
Finish random_seed 16
Fitting 5 folds for each of 45 candidates, totalling 225 fits
best_params: {'max_features': 20, 'max_leaf_nodes': 50, 'n_estimators': 70}
best_score: 0.7253189965769191
Finish random_seed 24
Fitting 5 folds for each of 45 candidates, totalling 225 fits
best_params: {'max_features': 20, 'max_leaf_nodes': 50, 'n_estimators': 90}
best_score: 0.728706671143976
Finish random_seed 32
Fitting 5 folds for each of 45 candidates, totalling 225 fits
best_params: {'max_features': 20, 'max_leaf_nodes': 50, 'n_estimators': 70}
best_score: 0.72919266526821
Finish random_seed 48
Fitting 5 folds for each of 45 candidates, tot

In [21]:
random_seed = 64
tr_x,te_x,tr_y,te_y = load_dataset(random_seed)
estimator = RandomForestClassifier(
    max_features=20,
    max_leaf_nodes=50,
    n_estimators=50,
    random_state=random_seed)
best_model = estimator.fit(tr_x, tr_y)
tr_result,te_result = Predict_Evalute(best_model,tr_x,te_x,tr_y,te_y,random_seed,fp='ecfp',model_name = 'RF')
joblib.dump(best_model,r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results\{fp}_{model}_seed_{random_seed}.joblib".format(fp='ecfp',model='RF',random_seed=random_seed))

['E:\\Projects\\Mycobacterium tuberculosis\\ML performance comparison\\Results\\ecfp_RF_seed_64.joblib']

In [35]:
random_seed = 64
tr_x,te_x,tr_y,te_y = load_dataset(random_seed)
estimator = xgb.XGBClassifier(
    colsample_bytree=0.5,
    learning_rate=0.1,
    max_depth=4,
    n_estimators=90,
    subsample=0.3,
    random_state=random_seed
)
best_model = estimator.fit(tr_x, tr_y)
tr_result,te_result = Predict_Evalute(best_model,tr_x,te_x,tr_y,te_y,random_seed,fp='ecfp',model_name = 'XGBoost')
joblib.dump(best_model,r"E:\Projects\Mycobacterium tuberculosis\ML performance comparison\Results\{fp}_{model}_seed_{random_seed}.joblib".format(fp='ecfp',model='XGBoost',random_seed=random_seed))

['E:\\Projects\\Mycobacterium tuberculosis\\ML performance comparison\\Results\\ecfp_XGBoost_seed_64.joblib']